# Spark default configuration

Lists **every** configuration parameter a Spark session has, with its default value and the value the session actually uses.

Spark exposes configuration in three places, each showing a different part:

| Source | What it returns |
|---|---|
| `spark.sparkContext.getConf().getAll()` | Only the parameters **explicitly set** (builder, `spark-defaults.conf`, launcher). |
| `spark.sql("SET -v")` | All **public SQL** parameters (`spark.sql.*`) with their current value and documentation. |
| `ConfigEntry.listAllEntries()` (JVM) | **All** registered parameters (core, SQL, internal), with default, doc and version. |

The session is created with the pipeline's `create_spark_session`, so it matches what `benchmark.py` / `validate.py` start from. The Spark UI is pinned to port 4040, so stop any other Spark session first.

In [1]:
import sys

import pandas as pd

sys.path.insert(0, "../pipeline")
from spark_session import ADAPTIVE_SETTINGS, BROADCAST_THRESHOLD_SETTING, create_spark_session

pd.set_option("display.max_rows", 50)
pd.set_option("display.max_colwidth", 120)

spark = create_spark_session("default_config")
spark.sparkContext.setLogLevel("ERROR")
spark.version

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/27 10:27:32 WARN Utils: Your hostname, bbas, resolves to a loopback address: 127.0.1.1; using 192.168.68.64 instead (on interface wlp0s20f3)
26/10/27 10:27:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/basbenja/DEAI/data-warehouses/project/part1/repo/.venv/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/27 10:27:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


'4.2.0'

## 1. Explicitly set parameters
Set by the builder in `create_spark_session` or by Spark at startup (driver host, app id, ...).

In [2]:
explicit = dict(spark.sparkContext.getConf().getAll())
explicit_df = pd.DataFrame(sorted(explicit.items()), columns=["key", "value"])
print(f"{len(explicit_df)} explicitly set parameters")
explicit_df

23 explicitly set parameters


,key,value
0,spark.app.id,local-1793093255810
1,spark.app.name,default_config
2,spark.app.startTime,1793093253365
3,spark.app.submitTime,1793093253075
4,spark.driver.extraJavaOptions,-Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-modules=jdk.incubator.vector --add-opens...
5,spark.driver.host,192.168.68.64
6,spark.driver.memory,6g
7,spark.driver.port,46403
8,spark.executor.extraJavaOptions,-Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-modules=jdk.incubator.vector --add-opens...
9,spark.executor.id,driver


## 2. Public SQL parameters (`SET -v`)
Current value of every documented `spark.sql.*` parameter, including those left at their default.

In [3]:
sql_df = spark.sql("SET -v").toPandas()
print(f"{len(sql_df)} public SQL parameters")
sql_df

/home/basbenja/DEAI/data-warehouses/project/part1/repo/.venv/lib/python3.13/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


298 public SQL parameters


,key,value,meaning,Since version
0,spark.sql.adaptive.advisoryPartitionSizeInBytes,<value of spark.sql.adaptive.shuffle.targetPostShuffleInputSize>,The advisory size in bytes of the shuffle partition during adaptive optimization (when spark.sql.adaptive.enabled is...,3.0.0
1,spark.sql.adaptive.autoBroadcastJoinThreshold,<undefined>,Configures the maximum size in bytes for a table that will be broadcast to all worker nodes when performing a join. ...,3.2.0
2,spark.sql.adaptive.coalescePartitions.enabled,true,"When true and 'spark.sql.adaptive.enabled' is true, Spark will coalesce contiguous shuffle partitions according to t...",3.0.0
3,spark.sql.adaptive.coalescePartitions.initialPartitionNum,<undefined>,"The initial number of shuffle partitions before coalescing. If not set, it equals to spark.sql.shuffle.partitions. T...",3.0.0
4,spark.sql.adaptive.coalescePartitions.minPartitionSize,1MB,The minimum size of shuffle partitions after coalescing. This is useful when the adaptively calculated target size i...,3.2.0
...,...,...,...,...
293,spark.sql.variable.substitute,true,"This enables substitution using syntax like `${var}`, `${system:var}`, and `${env:var}`.",2.0.0
294,spark.sql.warehouse.dir,file:/home/basbenja/DEAI/data-warehouses/project/part1/repo/notebooks/spark-warehouse,The default location for managed databases and tables.,2.0.0
295,spark.sql.window.segmentTree.blockSize,65536,"Block size, in rows, for the block-chunked segment tree used by moving window frames. Each leaf of the tree aggregat...",4.2.0
296,spark.sql.window.segmentTree.enabled,false,Use block-chunked segment tree for moving aggregate window frames whose functions are all DeclarativeAggregate witho...,4.2.0


## 3. All registered parameters
Every `ConfigEntry` known to the JVM. `effective_value` is what this session uses: the explicitly set value if any, otherwise the default. Parameters without a default show `<undefined>`.

In [4]:
def effective_value(key: str, default: str) -> str:
    if key in explicit:
        return explicit[key]
    try:
        return spark.conf.get(key)
    except Exception:
        return default


rows = []
entries = spark._jvm.org.apache.spark.internal.config.ConfigEntry.listAllEntries()
iterator = entries.iterator()
while iterator.hasNext():
    entry = iterator.next()
    key = entry.key()
    default = entry.defaultValueString()
    rows.append(
        {
            "key": key,
            "default": default,
            "effective_value": effective_value(key, default),
            "explicitly_set": key in explicit,
            "public": entry.isPublic(),
            "since": entry.version(),
            "doc": entry.doc(),
        }
    )

all_df = pd.DataFrame(rows).sort_values("key").reset_index(drop=True)
print(
    f"{len(all_df)} parameters "
    f"({all_df['public'].sum()} public, {(~all_df['public']).sum()} internal)"
)
all_df

1268 parameters (737 public, 531 internal)


,key,default,effective_value,explicitly_set,public,since,doc
0,spark.acls.enable,false,false,False,True,1.1.0,
1,spark.admin.acls,,,False,True,1.1.0,
2,spark.admin.acls.groups,,,False,True,2.0.0,
3,spark.api.mode,classic,classic,False,True,4.0.0,"For Spark Classic applications, specify whether to automatically use Spark Connect by running a local Spark Connect ..."
4,spark.app.attempt.id,<undefined>,<undefined>,False,False,3.2.0,"The application attempt Id assigned from Hadoop YARN. When the application runs in cluster mode on YARN, there can b..."
...,...,...,...,...,...,...,...
1263,spark.yarn.dist.pyFiles,,,False,False,2.2.1,
1264,spark.yarn.isPython,false,false,False,False,1.5.0,
1265,spark.yarn.kerberos.renewal.excludeHadoopFileSystems,,,False,True,3.2.0,The list of Hadoop filesystem URLs whose hosts will be excluded from delegation token renewal at resource scheduler....
1266,spark.yarn.shuffle.server.recovery.disabled,false,false,False,False,3.5.0,Set to true for applications that prefer to disable recovery when the External Shuffle Service restarts. This config...


### Parameters whose effective value differs from the default

In [5]:
all_df[all_df["effective_value"] != all_df["default"]]

,key,default,effective_value,explicitly_set,public,since,doc
42,spark.driver.extraJavaOptions,<undefined>,-Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-modules=jdk.incubator.vector --add-opens...,True,True,1.0.0,
53,spark.driver.memory,1g,6g,True,True,1.1.1,"Amount of memory to use for the driver process, in MiB unless otherwise specified."
58,spark.driver.port,0,46403,True,True,0.7.0,Port of driver endpoints.
112,spark.executor.extraJavaOptions,<undefined>,-Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-modules=jdk.incubator.vector --add-opens...,True,True,1.0.0,
118,spark.executor.id,<undefined>,driver,True,True,1.2.0,
...,...,...,...,...,...,...,...
1123,spark.sql.thriftserver.scheduler.pool,<undefined>,NaN,False,True,1.1.1,Set a Fair Scheduler pool for a JDBC client session.
1137,spark.sql.ui.explainMode,formatted,FORMATTED,False,True,3.1.0,"Configures the query explain mode used in the Spark SQL UI. The value can be 'simple', 'extended', 'codegen', 'cost'..."
1152,spark.sql.warehouse.dir,file:/home/basbenja/DEAI/data-warehouses/project/part1/repo/notebooks/spark-warehouse/,file:/home/basbenja/DEAI/data-warehouses/project/part1/repo/notebooks/spark-warehouse,False,True,2.0.0,The default location for managed databases and tables.
1158,spark.sql.windowExec.buffer.spill.size.threshold,<value of spark.shuffle.spill.maxSizeInBytesForSpillThreshold>,9223372036854775807b,False,False,4.1.0,Threshold for size of rows to be spilled by window operator


### Parameters driven by `config.toml`
The settings `apply_run_options` changes on the `changed` session and `reset_run_options` reverts on the baseline session.

In [6]:
run_option_keys = [*ADAPTIVE_SETTINGS, BROADCAST_THRESHOLD_SETTING]
all_df[all_df["key"].isin(run_option_keys)][["key", "default", "effective_value", "doc"]]

,key,default,effective_value,doc
397,spark.sql.adaptive.coalescePartitions.enabled,true,true,"When true and 'spark.sql.adaptive.enabled' is true, Spark will coalesce contiguous shuffle partitions according to t..."
403,spark.sql.adaptive.enabled,true,true,"When true, enable adaptive query execution, which re-optimizes the query plan in the middle of query execution, base..."
415,spark.sql.adaptive.skewJoin.enabled,true,true,"When true and 'spark.sql.adaptive.enabled' is true, Spark dynamically handles skew in shuffled join (sort-merge and ..."
457,spark.sql.autoBroadcastJoinThreshold,10MB,10485760b,Configures the maximum size in bytes for a table that will be broadcast to all worker nodes when performing a join. ...


### Search by key

In [7]:
pattern = "adaptive"
all_df[all_df["key"].str.contains(pattern, case=False)]

,key,default,effective_value,explicitly_set,public,since,doc
394,spark.sql.adaptive.advisoryPartitionSizeInBytes,<value of spark.sql.adaptive.shuffle.targetPostShuffleInputSize>,67108864b,False,True,3.0.0,The advisory size in bytes of the shuffle partition during adaptive optimization (when spark.sql.adaptive.enabled is...
395,spark.sql.adaptive.applyFinalStageShuffleOptimizations,true,true,False,False,3.4.2,Configures whether adaptive query execution (if enabled) should apply shuffle coalescing and local shuffle read opti...
396,spark.sql.adaptive.autoBroadcastJoinThreshold,<undefined>,NaN,False,True,3.2.0,Configures the maximum size in bytes for a table that will be broadcast to all worker nodes when performing a join. ...
397,spark.sql.adaptive.coalescePartitions.enabled,true,true,False,True,3.0.0,"When true and 'spark.sql.adaptive.enabled' is true, Spark will coalesce contiguous shuffle partitions according to t..."
398,spark.sql.adaptive.coalescePartitions.initialPartitionNum,<undefined>,NaN,False,True,3.0.0,"The initial number of shuffle partitions before coalescing. If not set, it equals to spark.sql.shuffle.partitions. T..."
399,spark.sql.adaptive.coalescePartitions.minPartitionNum,<undefined>,NaN,False,False,3.0.0,"(deprecated) The suggested (not guaranteed) minimum number of shuffle partitions after coalescing. If not set, the d..."
400,spark.sql.adaptive.coalescePartitions.minPartitionSize,1MB,1048576b,False,True,3.2.0,The minimum size of shuffle partitions after coalescing. This is useful when the adaptively calculated target size i...
401,spark.sql.adaptive.coalescePartitions.parallelismFirst,true,true,False,True,3.2.0,"When true, Spark does not respect the target size specified by 'spark.sql.adaptive.advisoryPartitionSizeInBytes' (de..."
402,spark.sql.adaptive.customCostEvaluatorClass,<undefined>,NaN,False,True,3.2.0,"The custom cost evaluator class to be used for adaptive execution. If not being set, Spark will use its own SimpleCo..."
403,spark.sql.adaptive.enabled,true,true,False,True,1.6.0,"When true, enable adaptive query execution, which re-optimizes the query plan in the middle of query execution, base..."


In [ ]:
spark.stop()